In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from discrete_action_space.pommerman_ffa.notebook_utils import (
    BASE_SEED,
    DEFAULT_NFG_TRANSFORMER_CHECKPOINT,
    RandomPolicy,
    baseline_evaluation_dir,
    configure_notebook_imports,
    evaluate_policy,
    evaluate_simple_agent_reference,
    evaluate_pommerman_deepsrq_nfg_transformer_for_epsilon,
    load_pommerman_iql_agent,
    load_pommerman_ippo_agent,
    plot_evaluation_rewards,
    policy_from_iql,
    policy_from_ippo,
)

REPO_ROOT = configure_notebook_imports()
OUTPUT_ROOT = baseline_evaluation_dir(repo_root=REPO_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPO_ROOT, OUTPUT_ROOT

In [ ]:
SMOKE_TEST = True

EPSILON = 0.5
N_EPISODES = 5 if SMOKE_TEST else 50
MAX_STEPS = 80 if SMOKE_TEST else 200
USE_GPU = True
SEED = BASE_SEED + 10_000
CHECKPOINT_NAME = "best"

NFG_TRANSFORMER_CHECKPOINT = DEFAULT_NFG_TRANSFORMER_CHECKPOINT
NFG_FALLBACK_ENABLED = False

config = {
    "epsilon": EPSILON,
    "n_episodes": N_EPISODES,
    "max_steps": MAX_STEPS,
    "checkpoint_name": CHECKPOINT_NAME,
    "output_root": str(OUTPUT_ROOT),
}
config

In [ ]:
eval_stats = {}

try:
    eval_stats["Deep SRQ + NfgTransformer"] = evaluate_pommerman_deepsrq_nfg_transformer_for_epsilon(
        EPSILON,
        n_episodes=N_EPISODES,
        max_steps=MAX_STEPS,
        seed=SEED,
        repo_root=REPO_ROOT,
        use_gpu=USE_GPU,
        checkpoint_name=CHECKPOINT_NAME,
        nfg_transformer_checkpoint_path=NFG_TRANSFORMER_CHECKPOINT,
        fallback_enabled=NFG_FALLBACK_ENABLED,
        verbose=True,
    )
except FileNotFoundError as exc:
    print(f"[Deep SRQ skipped: {exc}]")

eval_stats.keys()

In [ ]:
random_policy = RandomPolicy()
eval_stats["Random"] = evaluate_policy(
    lambda obs, order, episode, step: {name: random_policy.act(obs[name]) for name in order},
    n_episodes=N_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 1_000,
    output_dir=OUTPUT_ROOT / "random",
    label="random",
    verbose=True,
)

eval_stats["SimpleAgent"] = evaluate_simple_agent_reference(
    n_episodes=N_EPISODES,
    max_steps=MAX_STEPS,
    seed=SEED + 2_000,
    output_dir=OUTPUT_ROOT / "simple_agent",
    verbose=True,
)

try:
    iql_agent = load_pommerman_iql_agent(repo_root=REPO_ROOT, use_gpu=USE_GPU, checkpoint_name=CHECKPOINT_NAME)
    eval_stats["IQL-DQN"] = evaluate_policy(
        policy_from_iql(iql_agent),
        n_episodes=N_EPISODES,
        max_steps=MAX_STEPS,
        seed=SEED + 3_000,
        output_dir=OUTPUT_ROOT / "iql_dqn",
        label="iql_dqn",
        verbose=True,
    )
except FileNotFoundError as exc:
    print(f"[IQL-DQN skipped: {exc}]")

try:
    ippo_agent = load_pommerman_ippo_agent(repo_root=REPO_ROOT, use_gpu=USE_GPU, checkpoint_name=CHECKPOINT_NAME)
    eval_stats["IPPO"] = evaluate_policy(
        policy_from_ippo(ippo_agent),
        n_episodes=N_EPISODES,
        max_steps=MAX_STEPS,
        seed=SEED + 4_000,
        output_dir=OUTPUT_ROOT / "ippo",
        label="ippo",
        verbose=True,
    )
except FileNotFoundError as exc:
    print(f"[IPPO skipped: {exc}]")

list(eval_stats)

In [ ]:
rows = []
for label, stats in eval_stats.items():
    rewards = np.asarray(stats["episode_rewards"], dtype=np.float64)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    joint_rewards = rewards.sum(axis=1)
    rows.append(
        {
            "algorithm": label,
            "episodes": int(rewards.shape[0]),
            "mean_joint_reward": float(joint_rewards.mean()),
            "std_joint_reward": float(joint_rewards.std()),
            "mean_episode_length": float(np.mean(stats["episode_lengths"])),
        }
    )

rows

In [ ]:
labels = [row["algorithm"] for row in rows]
means = [row["mean_joint_reward"] for row in rows]
stds = [row["std_joint_reward"] for row in rows]

fig, ax = plt.subplots(figsize=(max(7, 1.6 * len(labels)), 4))
ax.bar(labels, means, yerr=stds, capsize=4)
ax.set_ylabel("Mean joint reward")
ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "algorithm_comparison_joint_reward.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
for label, stats in eval_stats.items():
    print(label)
    plot_evaluation_rewards(stats)